# Generating a CE dataset using MACE

In this notebook, we will learn how to create and compute a dataset for a Cluster Expansion model. We will be using a MACE model for that purpose to speed up the dataset generation. Everything is done with ASE calculators. If DFT is meant to be used, it should be easy to adjust the workflow.

To realize this, we will use the `icet` package. For a detailed description, see the corresponding paper [here](https://onlinelibrary.wiley.com/doi/abs/10.1002/adts.201900015) and the documentation [here](https://icet.materialsmodeling.org/). Several more tutorials provided by the developers can be found [here](https://ce-tutorials.materialsmodeling.org/). There are other solutions available, most notably [smol](https://github.com/CederGroupHub/smol). For basic applications both tools should work well, they mostly differ in their more advanced features (consult the respective package documentations in detail).

For this exercise, you can choose your favorite pair of FCC metals. Every notebook sets the variable `chemical_symbols` at the top. Change this to the chosen elements before running it. Check materials project whether these metals actually form an FCC structure first. If it is not the equilibrium structure, the code will probably still run, but reasonable results cannot be guaranteed. CE models will also be more accurate if the volume difference between the pure phase cells is small. Nonetheless, feel free to experiment.

Note, that in principle it is possible to specify multiple sublattice. For the example for 2 atoms in the base cell, we can specify `chemical_symbols = [["Mn","V"],["O","F"]]`, which seperates the allowed species for atom 1 and atom 2. It is also possible to simulate vacancies by specifying a "dummy" element representing the vacancy and removing those dummy elements when computing the reference data. However, for both of these cases, the code in this workshop would have to be adapted. See the more advanced icet tutorials for guidance.

## Load the MACE model

We are using the OMAT-0_small foundation model here. We will do no validation of its accuracy in the following. For an actual study, proper validation is essential.

Since the computation of the dataset takes a few tens of minutes depending on the machine, we provide precomputed datasets for CuNi and CuAu in the directory `data/CE_precomputed_datasets`. If you wish to use those, create the respective directories `data/CE_dataset_CuNi` or `data/CE_dataset_CuAu` and copy the relevant files there.

In [ ]:
from pathlib import Path
from mace.calculators import MACECalculator
import os
import torch
import urllib.request

base_path = Path.cwd().parents[1]
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# Change this to your favorite pair of elements forming an FCC crystal structure
chemical_symbols = ["Cu", "Au"]

data_target_dir = base_path / "data" / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
os.makedirs(data_target_dir, exist_ok=True)
# we use OMAT-0 small for our reference data generation. It is important to select an appropriate model for the appropriate application
mace_model_file_name = base_path / "data" / "models" / "OMAT0_small.model"

if not mace_model_file_name.exists():
    model_link = "https://github.com/ACEsuit/mace-foundations/releases/download/mace_omat_0/mace-omat-0-small.model"
    print(f"Downloading model to {mace_model_file_name}...")
    urllib.request.urlretrieve(model_link, mace_model_file_name)

calc = MACECalculator(mace_model_file_name, device=device)

## Relaxing the starting structures

We need to obtain reference energies for both materials to compute an energy of mixing. It is essential for CE to predict relative per-atom quantities. For this, we optimize the crystal structure for each of the phases.

The energy of mixing is a commonly used quantity. Depending on the application it can also be reasonable (or even necessary) to use the formation energy directly (where we would use the isolated atoms as a reference).

In [ ]:
import ase.io
from ase.atoms import Atoms
from ase.filters import UnitCellFilter
from ase.optimize import LBFGS
from ase.build import bulk

# We create the bulk structure. By default, ASE will asign some reference lattice parameter.
# We don't really care how accurate it is as we are going to relax the structure anyway.
prim_1 = bulk(chemical_symbols[0], "fcc")
prim_2 = bulk(chemical_symbols[1], "fcc")

def relax_structure(atoms: Atoms, output_file_name: str):
    atoms.calc = calc
    filter = UnitCellFilter(atoms)
    optimizer = LBFGS(
        filter,
        trajectory=output_file_name + ".traj",
        logfile=output_file_name + ".log",
    )
    converged = optimizer.run(fmax=0.0001)
    ase.io.write(output_file_name + ".extxyz", atoms, format="extxyz")
    return atoms, converged


relaxed_1, _ = relax_structure(prim_1, f'{data_target_dir / f"{chemical_symbols[0]}_relaxed"}')
relaxed_2, _ = relax_structure(prim_2, f'{data_target_dir / f"{chemical_symbols[1]}_relaxed"}')

ref_energies = {}
ref_energies[chemical_symbols[0]] = relaxed_1.get_potential_energy() / len(relaxed_1)
ref_energies[chemical_symbols[1]] = relaxed_2.get_potential_energy() / len(relaxed_2)

for chemical_symbol in chemical_symbols:
    print(f"{chemical_symbol} reference energy: {ref_energies[chemical_symbol]} eV/atom")

## Enumeration of structures

First, we need to build the primitive structure for CE. We just take one of the FCC base crystals and allow swaps with Au by setting the appropriate chemical symbols. Then we will create ALL possible symmetry-inequivalent combinations up to `max_num_atoms` in size. For a simple 2 component system such as this, that approach is reasonable. For a more complex base lattice or one containing multiple options the number of elements in the sub-lattice, this will quickly become unreasonable. For such systems, structures need to be generated (either randomly or via simulation methods like Monte Carlo) and subsequently selected using some kind of algorithm. Possible options are optimizing condition numbers, pure random sampling, or uncertainty-based selection. For details, see [this paper](https://link.aps.org/doi/10.1103/PRXEnergy.3.042001), which also includes tutorial notebooks for the different training methods. In this exercise, we will limit ourselves to simple enumeration.

In [ ]:
from icet.tools import enumerate_structures

max_num_atoms = 8
primitive_structure = relaxed_1

# by default this will exclude symmetrically equivalent structures
enumerated_structures = list(
    enumerate_structures(
        structure=primitive_structure,
        sizes=range(1, max_num_atoms + 1),
        chemical_symbols=chemical_symbols,
        niggli_reduce=False
    )
)

ase.io.write(
    data_target_dir / f"enumerated_structures_{max_num_atoms}.extxyz",
    enumerated_structures,
)
print("number of generated structures:", len(enumerated_structures))

## Run MACE relaxations for each of these structures

The CE model can only predict a quantity (in our case the mixing energy) for an ideal lattice. However, we want it to predict the energy for the respective equilibrium structures. Therefore obtain the reference data, we need to optimize the atomic positions and lattice parameters for every generated structure. For training, we will then use the pristine unrelaxed structure in combination with the mixing energy per atom (using a per atom quantity is vital here due to implementation of `icet`). Traditionally DFT is used to obtain the reference data. However, foundation models are rather good these days and in this course we just create a surrogate model for the surrogate model.

Note, that relaxation runs themselves can also sometimes have difficulties to converge. It can also be possible that no convergence is achieved. Here, we set a maximum number of relaxation steps per structure. Also, sometimes the structures will relax to off-lattice sites which deviate so strongly from the "ideal" lattice that it is reasonable to discard the structure before training. 

Run time is approximately 10 min on an RTX 4050 for `max_num_atoms = 8`.

In [ ]:
from tqdm import tqdm
import numpy as np
from copy import copy

max_relax_steps = 300  # set the maximum relaxation steps, if convergence is not achieved within this time frame, the structure is discarded

relaxed_structures = []
converged_structures = np.zeros(len(enumerated_structures), dtype=bool)
for sid, struct in tqdm(enumerate(enumerated_structures), total=len(enumerated_structures)):
    struct.calc = calc
    unrelaxed_coordinates = copy(
        struct.positions
    )  # we need to copy the coordinates as they will be modified during the relaxation
    unrelaxed_cell = copy(struct.cell.array)
    filter = UnitCellFilter(struct)
    optimizer = LBFGS(
        filter,
        logfile=None,
    )  # setting logfile to None suppresses the output
    converged = optimizer.run(
        fmax=0.0001,
        steps=max_relax_steps,
    )
    if converged:
        energy = struct.get_potential_energy()
        mixing_energy = energy
        for element in np.unique(struct.get_chemical_symbols()):
            mixing_energy -= ref_energies[element] * np.sum(
                struct.symbols == element
            )
        struct.info["mixing_energy"] = mixing_energy / len(struct)
        struct.calc = None  # we do this to not overwrite energy/forces entries of the atoms objects
        struct.info["energy"] = energy
        relaxed_structures.append(struct.copy())
        # we reset to the original positions/lattice for the training of the CE model
        struct.positions = unrelaxed_coordinates
        struct.set_cell(unrelaxed_cell)
        converged_structures[sid] = True
    else:
        print(f"WARNING: Structure {sid} did not converge")

# only include converged structures
converged_and_calculated = [
    struct
    for struct, converged in zip(enumerated_structures, converged_structures)
    if converged
]
ase.io.write(
    data_target_dir
    / f"enumerated_structures_{max_num_atoms}_calculated.extxyz",
    converged_and_calculated,
)
# we also output the relaxed structures for further observations
ase.io.write(
    data_target_dir
    / f"enumerated_structures_{max_num_atoms}_relaxed.extxyz",
    relaxed_structures,
)